# CHALLENGE: SQL Skills Challenge

This is the end-of-module challenge. Apply everything you’ve learned across all the demos: aggregations, window functions, QUALIFY, CTEs, JOINs, and views.

**Instructions:**
- Each task has a SQL cell with most of the code written for you
- Replace the `<FILL_IN>` placeholder(s) with the correct SQL to complete the query
- Run each cell to verify your answer
- Solutions are at the bottom — try each task first!

**Tables available:**
- `store_transactions` — 430 rows (60 days of sales, 3 regions, 4 product categories, includes ~30 duplicate rows)
- `product_catalog` — 4 rows (category reference with department and margin %)

---

In [0]:
%run ./Setup

## Task 1: Total Revenue by Region (GROUP BY + Aggregation)

Calculate the **total revenue** and **transaction count** for each region. Order results from highest revenue to lowest.

**Expected result:** 3 rows, one per region.

In [0]:
%sql
-- Task 1: Replace <FILL_IN> with the correct aggregate function and clause
SELECT 
  region,
  ROUND(<FILL_IN>(amount), 2) AS total_revenue,
  COUNT(*) AS transaction_count
FROM store_transactions
<FILL_IN> region
ORDER BY total_revenue DESC;

---

## Task 2: Filter Before Aggregating (WHERE + GROUP BY)

Find the total revenue for **only online transactions** (`channel = 'online'`), grouped by product category.

**Expected result:** 4 rows (one per product category), showing only online channel revenue.

In [0]:
%sql
-- Task 2: Replace <FILL_IN> with the correct filter condition
SELECT 
  product_category,
  ROUND(SUM(amount), 2) AS online_revenue,
  COUNT(*) AS online_transactions
FROM store_transactions
WHERE <FILL_IN>
GROUP BY product_category
ORDER BY online_revenue DESC;

---

## Task 3: Prior Week Comparison (LAG Window Function)

Calculate **weekly revenue** and add a column showing the **prior week’s revenue** using `LAG()`.

**Hint:** You need `DATE_TRUNC('week', txn_date)` to group by week, and `LAG(...)` to look back one row.

**Expected result:** ~9 rows. The first row’s `prior_week_revenue` will be NULL (no prior week exists).

In [0]:
%sql
-- Task 3: Replace <FILL_IN> with the window function that looks back one row
SELECT 
  DATE_TRUNC('week', txn_date) AS week_start,
  ROUND(SUM(amount), 2) AS weekly_revenue,
  ROUND(<FILL_IN>(SUM(amount)) OVER (ORDER BY DATE_TRUNC('week', txn_date)), 2) AS prior_week_revenue
FROM store_transactions
GROUP BY DATE_TRUNC('week', txn_date)
ORDER BY week_start;

---

## Task 4: Remove Duplicates (QUALIFY + ROW_NUMBER)

The `store_transactions` table has ~30 duplicate rows. Use `QUALIFY` with `ROW_NUMBER()` to deduplicate based on `transaction_id`, keeping only the first occurrence (order by `txn_date`).

**Expected result:** Exactly 400 rows (the 430 rows minus the ~30 duplicates).

In [0]:
%sql
-- Task 4: Replace <FILL_IN> to complete the deduplication
-- Hint: PARTITION BY the unique ID, ORDER BY the date
SELECT *
FROM store_transactions
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY <FILL_IN> 
  ORDER BY <FILL_IN>
) = 1;

---

## Task 5: 7-Day Rolling Average (Sliding Window)

Calculate the **daily total revenue** and a **7-day rolling average** across all regions.

**Hint:** Use `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` to define the sliding window frame.

**Expected result:** ~60 rows. The rolling average should smooth out daily fluctuations.

In [0]:
%sql
-- Task 6: Replace <FILL_IN> with the correct window frame clause
SELECT 
  txn_date,
  ROUND(SUM(amount), 2) AS daily_revenue,
  ROUND(AVG(SUM(amount)) OVER (
    ORDER BY txn_date
    <FILL_IN>
  ), 2) AS rolling_7day_avg
FROM store_transactions
GROUP BY txn_date
ORDER BY txn_date;

---

## Task 6: JOIN with Reference Table

Join `store_transactions` to `product_catalog` to calculate **estimated profit** (amount × margin_pct / 100) by region and department.

**Hint:** Join on the shared `product_category` column.

**Expected result:** 6 rows (3 regions × 2 departments).

In [0]:
%sql
-- Task 7: Replace <FILL_IN> with the correct JOIN type and condition
SELECT 
  t.region,
  p.department,
  ROUND(SUM(t.amount), 2) AS total_revenue,
  ROUND(SUM(t.amount * p.margin_pct / 100), 2) AS estimated_profit
FROM store_transactions t
<FILL_IN> product_catalog p 
  ON <FILL_IN>
GROUP BY t.region, p.department
ORDER BY estimated_profit DESC;

---

## Task 7: Multi-Step CTE

Build a CTE-based query that:
1. **CTE `daily_totals`**: calculates daily revenue by region
2. **CTE `with_avg`**: adds each region’s overall average daily revenue using a window function
3. **Final SELECT**: finds days where revenue exceeded 1.5× the region average

**Hint:** The structure is `WITH cte1 AS (...), cte2 AS (...) SELECT ... FROM cte2 WHERE ...`

**Expected result:** Several rows showing above-average days.

In [0]:
%sql
-- Task 8: Replace <FILL_IN> to complete the CTE query
WITH daily_totals AS (
  SELECT 
    txn_date,
    region,
    SUM(amount) AS daily_revenue
  FROM store_transactions
  GROUP BY txn_date, region
),
with_avg AS (
  SELECT 
    *,
    <FILL_IN>(daily_revenue) OVER (PARTITION BY region) AS region_avg
  FROM daily_totals
)
SELECT 
  txn_date, 
  region,
  ROUND(daily_revenue, 2) AS daily_revenue,
  ROUND(region_avg, 2) AS region_avg,
  ROUND(daily_revenue - region_avg, 2) AS vs_average
FROM with_avg
WHERE daily_revenue > region_avg * <FILL_IN>
ORDER BY vs_average DESC
LIMIT 10;

---

## Task 8: Create a Reusable View

Create a **standard view** called `v_clean_transactions` that:
1. Deduplicates using `QUALIFY`
2. Adds a calculated `line_total` column (amount × quantity)
3. Only includes transactions from March 2024 onwards

**Expected result:** The view is created successfully. Query it to verify.

In [0]:
%sql
-- Task 9: Replace <FILL_IN> to create the view
<FILL_IN> v_clean_transactions AS
SELECT 
  transaction_id,
  txn_date,
  region,
  product_category,
  channel,
  amount,
  quantity,
  ROUND(amount * quantity, 2) AS line_total
FROM store_transactions
WHERE txn_date >= '2024-03-01'
QUALIFY ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY txn_date) = 1;

In [0]:
%sql
-- Verify your view works
SELECT region, COUNT(*) AS rows, ROUND(SUM(line_total), 2) AS total
FROM v_clean_transactions
GROUP BY region
ORDER BY total DESC;

---
---
---

## ⬇️ Solutions (Try the tasks first!) ⬇️

Scroll down only after you’ve attempted all tasks above.

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### Solution 1

Replace with `SUM` and `GROUP BY`.

In [0]:
%sql
SELECT 
  region,
  ROUND(SUM(amount), 2) AS total_revenue,
  COUNT(*) AS transaction_count
FROM store_transactions
GROUP BY region
ORDER BY total_revenue DESC;

### Solution 2

Replace with `channel = 'online'`.

In [0]:
%sql
SELECT 
  product_category,
  ROUND(SUM(amount), 2) AS online_revenue,
  COUNT(*) AS online_transactions
FROM store_transactions
WHERE channel = 'online'
GROUP BY product_category
ORDER BY online_revenue DESC;

### Solution 3

Replace with `LAG` — it looks back one row in the ordered window.

In [0]:
%sql
SELECT 
  DATE_TRUNC('week', txn_date) AS week_start,
  ROUND(SUM(amount), 2) AS weekly_revenue,
  ROUND(LAG(SUM(amount)) OVER (ORDER BY DATE_TRUNC('week', txn_date)), 2) AS prior_week_revenue
FROM store_transactions
GROUP BY DATE_TRUNC('week', txn_date)
ORDER BY week_start;

### Solution 4

Replace with `transaction_id` (PARTITION BY) and `txn_date` (ORDER BY). Result: 400 rows.

In [0]:
%sql
SELECT *
FROM store_transactions
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY transaction_id 
  ORDER BY txn_date
) = 1;

### Solution 5

Replace with `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` (6 prior + current = 7 days).

In [0]:
%sql
SELECT 
  txn_date,
  ROUND(SUM(amount), 2) AS daily_revenue,
  ROUND(AVG(SUM(amount)) OVER (
    ORDER BY txn_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ), 2) AS rolling_7day_avg
FROM store_transactions
GROUP BY txn_date
ORDER BY txn_date;

### Solution 6

Replace with `INNER JOIN` and `t.product_category = p.product_category`.

In [0]:
%sql
SELECT 
  t.region,
  p.department,
  ROUND(SUM(t.amount), 2) AS total_revenue,
  ROUND(SUM(t.amount * p.margin_pct / 100), 2) AS estimated_profit
FROM store_transactions t
INNER JOIN product_catalog p 
  ON t.product_category = p.product_category
GROUP BY t.region, p.department
ORDER BY estimated_profit DESC;

### Solution 7

Replace with `AVG` (for the window function) and `1.5` (for the threshold multiplier). The CTE chains daily aggregation → adds region average → filters outliers.

In [0]:
%sql
WITH daily_totals AS (
  SELECT 
    txn_date,
    region,
    SUM(amount) AS daily_revenue
  FROM store_transactions
  GROUP BY txn_date, region
),
with_avg AS (
  SELECT 
    *,
    AVG(daily_revenue) OVER (PARTITION BY region) AS region_avg
  FROM daily_totals
)
SELECT 
  txn_date, 
  region,
  ROUND(daily_revenue, 2) AS daily_revenue,
  ROUND(region_avg, 2) AS region_avg,
  ROUND(daily_revenue - region_avg, 2) AS vs_average
FROM with_avg
WHERE daily_revenue > region_avg * 1.5
ORDER BY vs_average DESC
LIMIT 10;

### Solution 8

Replace with `CREATE OR REPLACE VIEW`. The view combines deduplication, a calculated column, and a date filter in one reusable object.

In [0]:
%sql
CREATE OR REPLACE VIEW v_clean_transactions AS
SELECT 
  transaction_id,
  txn_date,
  region,
  product_category,
  channel,
  amount,
  quantity,
  ROUND(amount * quantity, 2) AS line_total
FROM store_transactions
WHERE txn_date >= '2024-03-01'
QUALIFY ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY txn_date) = 1;

In [0]:
%sql
-- Verify
SELECT region, COUNT(*) AS rows, ROUND(SUM(line_total), 2) AS total
FROM v_clean_transactions
GROUP BY region
ORDER BY total DESC;